# WNBA Knowledge Graph 

Dataset: 763 pemain WNBA, 20 tim, 143 universitas, 9 posisi (`dataset_WNBA.csv`).


In [1]:
import sys
sys.path.append("../src")
import pandas as pd


## Dataset

In [3]:
df = pd.read_csv("/Users/byg/Desktop/GRAF UAS WNBA/WNBA-Graph-FinalProject/data/dataset_WNBA.csv")
print(df.shape)
df.head()


(1328, 6)


,playerLabel,teamLabel,collegeLabel,positionLabel,height,draftYear
0,Megan Gustafson,Las Vegas Aces,University of Iowa,center,75.0,2019.0
1,Awak Kuier,Dallas Wings,NaN,power forward,195.0,2021.0
2,Aliyah Boston,Indiana Fever,University of South Carolina,NaN,196.0,2023.0
3,Jackie Young,Las Vegas Aces,University of Notre Dame,shooting guard,183.0,2019.0
4,Sabrina Ionescu,New York Liberty,University of Oregon,point guard,180.0,2020.0


In [4]:
print("Jumlah pemain       :", df['playerLabel'].nunique())
print("Jumlah tim          :", df['teamLabel'].nunique())
print("Jumlah universitas  :", df['collegeLabel'].nunique())
print("Posisi              :", df['positionLabel'].dropna().unique())


Jumlah pemain       : 763
Jumlah tim          : 20
Jumlah universitas  : 143
Posisi              : ['center' 'power forward' 'shooting guard' 'point guard' 'small forward'
 'forward' 'guard' 'combo guard']


## 1. Koneksi Database 

Membuktikan koneksi ke Neo4j berhasil dan menampilkan ringkasan jumlah node per label.


In [5]:
from db import get_connection

conn = get_connection()
print("Koneksi berhasil:", conn.verify_connectivity())

summary = conn.run(
    "MATCH (n) RETURN labels(n)[0] AS label, count(*) AS jumlah ORDER BY jumlah DESC"
)
pd.DataFrame(summary)


Koneksi berhasil: True


,label,jumlah
0,Player,763
1,College,143
2,Team,20
3,Position,8
4,Award,1
5,Event,1


## 2. Graph Analytics 



In [ ]:
# Memproyeksikan graf ke memori GDS secara transparan
print("Memeriksa dan membersihkan memori graf lama...")
try:
    conn.run("CALL gds.graph.drop('wnbaGraph')")
except Exception:
    pass # Abaikan jika graf memang belum ada

print("Memproyeksikan graf baru ke memori...")
project_query = '''
CALL gds.graph.project(
  'wnbaGraph',
  ['Player', 'Team', 'College'],
  {
    PLAYS_FOR: {orientation: 'UNDIRECTED'},
    EDUCATED_AT: {orientation: 'UNDIRECTED'}
  }
)
'''
hasil = conn.run(project_query)
print("Berhasil! Detail proyeksi:")
pd.DataFrame(hasil)

Memeriksa dan membersihkan memori graf lama...
Memproyeksikan graf baru ke memori...
Berhasil! Detail proyeksi:


,nodeProjection,relationshipProjection,graphName,nodeCount,relationshipCount,projectMillis
0,"{'College': {'properties': {}, 'label': 'Colle...","{'EDUCATED_AT': {'orientation': 'UNDIRECTED', ...",wnbaGraph,926,3424,137


In [ ]:
# SIMILARITY: Menemukan dua tim yang memiliki pemain yang sama paling banyak
similarity_query = '''
MATCH (t1:Team)<-[:PLAYS_FOR]-(p:Player)-[:PLAYS_FOR]->(t2:Team)
WHERE id(t1) < id(t2)
RETURN t1.name AS Team_1, t2.name AS Team_2, 
       count(p) AS Shared_Players
ORDER BY Shared_Players DESC
LIMIT 10
'''
pd.DataFrame(conn.run(similarity_query))

,Team_1,Team_2,Shared_Players
0,Los Angeles Sparks,Washington Mystics,20
1,Phoenix Mercury,Seattle Storm,18
2,Phoenix Mercury,Minnesota Lynx,17
3,New York Liberty,Los Angeles Sparks,17
4,Seattle Storm,Minnesota Lynx,17
5,Washington Mystics,Minnesota Lynx,16
6,Los Angeles Sparks,Minnesota Lynx,16
7,New York Liberty,Washington Mystics,14
8,Atlanta Dream,Chicago Sky,14
9,Atlanta Dream,Washington Mystics,14


In [ ]:
# DEGREE CENTRALITY
degree_query = '''
CALL gds.degree.stream('wnbaGraph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS Name,
       labels(gds.util.asNode(nodeId))[0] AS Type,
       score AS Degree_Score
ORDER BY Degree_Score DESC
LIMIT 10
'''
pd.DataFrame(conn.run(degree_query))

,Name,Type,Degree_Score
0,Minnesota Lynx,Team,119.0
1,New York Liberty,Team,112.0
2,Los Angeles Sparks,Team,110.0
3,Washington Mystics,Team,106.0
4,Phoenix Mercury,Team,104.0
5,Seattle Storm,Team,90.0
6,Chicago Sky,Team,84.0
7,Connecticut Sun,Team,84.0
8,Indiana Fever,Team,81.0
9,Atlanta Dream,Team,74.0


In [ ]:
# PAGERANK
pagerank_query = '''
CALL gds.pageRank.stream('wnbaGraph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS Name,
       labels(gds.util.asNode(nodeId))[0] AS Type,
       score AS PageRank_Score
ORDER BY PageRank_Score DESC
LIMIT 10
'''
pd.DataFrame(conn.run(pagerank_query))

,Name,Type,PageRank_Score
0,Minnesota Lynx,Team,28.587146
1,New York Liberty,Team,26.153532
2,Los Angeles Sparks,Team,25.268177
3,Washington Mystics,Team,24.593405
4,Phoenix Mercury,Team,24.496784
5,Chicago Sky,Team,20.179764
6,Seattle Storm,Team,20.078440
7,Connecticut Sun,Team,20.001222
8,Indiana Fever,Team,18.045498
9,Atlanta Dream,Team,17.357130


In [ ]:
# LOUVAIN (Community Detection)
louvain_query = '''
CALL gds.louvain.stream('wnbaGraph')
YIELD nodeId, communityId
RETURN gds.util.asNode(nodeId).name AS Name,
       labels(gds.util.asNode(nodeId))[0] AS Type,
       communityId AS Community_ID
ORDER BY Community_ID ASC
LIMIT 15
'''
pd.DataFrame(conn.run(louvain_query))

,Name,Type,Community_ID
0,Amanda Zahui,Player,76
1,Fred Williams,Player,76
2,Glory Johnson,Player,76
3,Boise State University,College,76
4,University of Minnesota,College,76
5,Fairfield University,College,76
6,Awak Kuier,Player,76
7,Villanova University,College,76
8,Janel McCarville,Player,76
9,Tricia Bader Binford,Player,76


In [ ]:
# TRAVERSAL: pemain & tim yang terhubung lewat universitas yang sama
traversal_query = '''
MATCH (p1:Player)-[:EDUCATED_AT]->(c:College)<-[:EDUCATED_AT]-(p2:Player)
WHERE p1.name < p2.name
RETURN c.name AS college, p1.name AS player1, p2.name AS player2
LIMIT 10
'''
pd.DataFrame(conn.run(traversal_query))

,college,player1,player2
0,University of Iowa,Kate Martin,Megan Gustafson
1,University of Iowa,Caitlin Clark,Megan Gustafson
2,University of Iowa,Caitlin Clark,Kate Martin
3,University of South Carolina,A'ja Wilson,Aliyah Boston
4,University of South Carolina,Aliyah Boston,Shannon Johnson
5,University of South Carolina,A'ja Wilson,Shannon Johnson
6,University of South Carolina,Brian Winters,Shannon Johnson
7,University of South Carolina,Jocelyn Penn,Shannon Johnson
8,University of South Carolina,Allisha Gray,Shannon Johnson
9,University of South Carolina,Mikiah Herbert Harrigan,Shannon Johnson


## 3. LLM untuk Text-to-Cypher 

In [8]:
from text_to_cypher import ask

hasil = ask("Sebutkan 5 pemain dengan tinggi badan tertinggi beserta universitasnya")
print("Cypher yang dihasilkan LLM:\n", hasil["cypher"])
print()
print("Jawaban:", hasil["answer"])
pd.DataFrame(hasil["rows"])


Cypher yang dihasilkan LLM:
 MATCH (p:Player)-[:EDUCATED_AT]->(c:College)
WHERE p.height IS NOT NULL
WITH p, c, p.height AS tinggi
ORDER BY tinggi DESC
RETURN p.name AS Nama, c.name AS Universitas, tinggi AS Tinggi
LIMIT 5

Jawaban: Berikut 5 pemain dengan tinggi badan tertinggi beserta universitasnya: 

1. Tree Rollins dari Clemson University dengan tinggi 216 cm
2. Orlando Woolridge dari University of Notre Dame dengan tinggi 206 cm
3. Brittney Griner dari Baylor University dengan tinggi 203 cm
4. Katie Feenstra-Mattera dari Liberty University dengan tinggi 203 cm
5. Alison Bales dari Duke University tidak masuk dalam daftar karena tingginya hanya 201 cm.


,Nama,Universitas,Tinggi
0,Tree Rollins,Clemson University,216.0
1,Orlando Woolridge,University of Notre Dame,206.0
2,Brittney Griner,Baylor University,203.0
3,Katie Feenstra-Mattera,Liberty University,203.0
4,Alison Bales,Duke University,201.0


In [9]:
hasil2 = ask("Universitas mana yang paling banyak menghasilkan pemain?")
print(hasil2["cypher"])
print(hasil2["answer"])
pd.DataFrame(hasil2["rows"])


MATCH (p:Player)-[:EDUCATED_AT]->(c:College)
RETURN c.name AS Universitas, COUNT(p) AS Jumlah_Pemain
ORDER BY Jumlah_Pemain DESC
LIMIT 25
Universitas yang paling banyak menghasilkan pemain adalah University of Connecticut dengan jumlah pemain sebanyak 25.


,Universitas,Jumlah_Pemain
0,University of Connecticut,25
1,Stanford University,23
2,University of Tennessee,19
3,University of Florida,14
4,University of Notre Dame,14
5,University of South Carolina,12
6,Duke University,11
7,University of Virginia,9
8,University of Maryland,9
9,University of North Carolina at Chapel Hill,9


## 4. Machine Learning pada Graph (GDS)

FastRP node embedding → K-Means clustering pemain → klasifikasi posisi pemain
menggunakan embedding + tinggi badan.


In [10]:
from graph_ml import get_gds, project_graph, run_node_embedding, run_clustering, run_position_classification

gds = get_gds()
graph = project_graph(gds)
print(graph.node_count(), "node |", graph.relationship_count(), "relasi diproyeksikan ke GDS")


/Users/byg/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


934 node | 4736 relasi diproyeksikan ke GDS


In [11]:
run_node_embedding(gds, graph)
clusters = run_clustering(gds, n_clusters=5)
clusters.head(10)


,name,cluster
0,Megan Gustafson,3
1,Awak Kuier,0
2,Aliyah Boston,0
3,Jackie Young,3
4,Sabrina Ionescu,2
5,Lou Lopez Sénéchal,2
6,Kate Martin,3
7,Cameron Brink,2
8,Rickea Jackson,2
9,Rhyne Howard,2


In [12]:
report = run_position_classification(gds)
print(report)


                precision    recall  f1-score   support

        center       0.90      0.93      0.91        40
       forward       0.00      0.00      0.00         1
   point guard       0.88      0.92      0.90        25
 power forward       0.84      0.62      0.71        26
shooting guard       0.90      0.90      0.90        42
 small forward       0.77      0.93      0.84        29

      accuracy                           0.87       163
     macro avg       0.72      0.72      0.71       163
  weighted avg       0.86      0.87      0.86       163



In [13]:
graph.drop()
gds.close()


## 5. LLM for Graph Builder 

Ekstraksi entitas & relasi dari teks bebas (misal cuplikan berita/bio pemain),
lalu populate otomatis ke Neo4j. (Screenshot B)


In [14]:
from graph_builder import build_graph_from_text

teks_bebas = '''
A'ja Wilson bermain untuk Las Vegas Aces dan merupakan alumni dari
University of South Carolina. Pada tahun 2024, ia memenangkan WNBA
Most Valuable Player Award untuk ketiga kalinya dan membawa Las Vegas
Aces tampil di WNBA Finals.
'''

hasil_builder = build_graph_from_text(teks_bebas)
print(hasil_builder["extracted"])
print("Ditulis ke Neo4j:", hasil_builder["write_summary"])


{'entities': [{'type': 'Player', 'name': "A'ja Wilson"}, {'type': 'Team', 'name': 'Las Vegas Aces'}, {'type': 'College', 'name': 'University of South Carolina'}, {'type': 'Award', 'name': 'WNBA Most Valuable Player Award'}, {'type': 'Event', 'name': 'WNBA Finals'}], 'relations': [{'source': "A'ja Wilson", 'type': 'PLAYS_FOR', 'target': 'Las Vegas Aces'}, {'source': "A'ja Wilson", 'type': 'ATTENDED', 'target': 'University of South Carolina'}, {'source': "A'ja Wilson", 'type': 'WON', 'target': 'WNBA Most Valuable Player Award'}, {'source': 'Las Vegas Aces', 'type': 'PARTICIPATED_IN', 'target': 'WNBA Finals'}]}
Ditulis ke Neo4j: {'entities_written': 5, 'relations_written': 4}


## 6. RAG (Graph-Augmented Retrieval) 

Mengambil subgraph relevan sebagai konteks, lalu menjawab pertanyaan dengan Gemini
secara grounded pada data graph (mengurangi halusinasi). (Screenshot D)


In [15]:
from rag import rag_answer

res = rag_answer("Apa hubungan antara Paige Bueckers dan Dallas Wings?")
print("Konteks subgraph:\n", res["context"])
print()
print("Jawaban RAG:", res["answer"])


Konteks subgraph:
 Paige Bueckers -[PLAYS_FOR]- Dallas Wings (Team)
Paige Bueckers -[PLAYS_FOR -> PLAYS_FOR]- Megan Gustafson (Player)
Paige Bueckers -[PLAYS_FOR -> PLAYS_FOR]- Awak Kuier (Player)
Paige Bueckers -[PLAYS_FOR -> PLAYS_FOR]- Lou Lopez Sénéchal (Player)
Paige Bueckers -[PLAYS_FOR -> PLAYS_FOR]- Arike Ogunbowale (Player)
Paige Bueckers -[PLAYS_FOR -> PLAYS_FOR]- Maddy Siegrist (Player)
Paige Bueckers -[PLAYS_FOR -> PLAYS_FOR]- Evelyn Akhator (Player)
Paige Bueckers -[PLAYS_FOR -> PLAYS_FOR]- Glory Johnson (Player)
Paige Bueckers -[PLAYS_FOR -> PLAYS_FOR]- Amanda Zahui (Player)
Paige Bueckers -[PLAYS_FOR -> PLAYS_FOR]- Fred Williams (Player)
Paige Bueckers -[PLAYS_FOR -> PLAYS_FOR]- Aerial Powers (Player)
Paige Bueckers -[PLAYS_FOR -> PLAYS_FOR]- Marina Mabrey (Player)
Paige Bueckers -[PLAYS_FOR -> PLAYS_FOR]- Chrissy Givens (Player)
Paige Bueckers -[PLAYS_FOR -> PLAYS_FOR]- Allisha Gray (Player)
Paige Bueckers -[PLAYS_FOR -> PLAYS_FOR]- Breanna Lewis (Player)
Paige Bueckers

In [16]:
res2 = rag_answer("Pemain mana saja dari University of Iowa yang ada di graph ini?")
print(res2["context"])
print()
print(res2["answer"])

University of Oregon -[EDUCATED_AT]- Sabrina Ionescu (Player)
University of Oregon -[EDUCATED_AT -> PLAYS_FOR]- New York Liberty (Team)
University of Oregon -[EDUCATED_AT -> PLAYS_POSITION]- point guard (Position)
University of Oregon -[EDUCATED_AT -> ATTENDED]- University of Oregon (College)
University of Oregon -[EDUCATED_AT]- Ruthy Hebard (Player)
University of Oregon -[EDUCATED_AT -> PLAYS_FOR]- Chicago Sky (Team)
University of Oregon -[EDUCATED_AT]- Cathrine Kraayeveld (Player)
University of Oregon -[EDUCATED_AT -> PLAYS_FOR]- Atlanta Dream (Team)
University of Oregon -[EDUCATED_AT -> PLAYS_POSITION]- small forward (Position)
University of Oregon -[EDUCATED_AT]- Maite Cazorla (Player)
University of Oregon -[EDUCATED_AT]- Satou Sabally (Player)
University of Oregon -[EDUCATED_AT -> PLAYS_FOR]- Dallas Wings (Team)
University of Oregon -[EDUCATED_AT -> PLAYS_FOR]- Phoenix Mercury (Team)
University of Oregon -[ATTENDED]- Sabrina Ionescu (Player)
University of Oregon -[ATTENDED -> PLAY

In [17]:
res2 = rag_answer("Pemain mana saja dari University of Connecticut yang ada di graph ini?")
print(res2["context"])
print()
print(res2["answer"])


University of Oregon -[EDUCATED_AT]- Sabrina Ionescu (Player)
University of Oregon -[EDUCATED_AT -> PLAYS_FOR]- New York Liberty (Team)
University of Oregon -[EDUCATED_AT -> PLAYS_POSITION]- point guard (Position)
University of Oregon -[EDUCATED_AT -> ATTENDED]- University of Oregon (College)
University of Oregon -[EDUCATED_AT]- Ruthy Hebard (Player)
University of Oregon -[EDUCATED_AT -> PLAYS_FOR]- Chicago Sky (Team)
University of Oregon -[EDUCATED_AT]- Cathrine Kraayeveld (Player)
University of Oregon -[EDUCATED_AT -> PLAYS_FOR]- Atlanta Dream (Team)
University of Oregon -[EDUCATED_AT -> PLAYS_POSITION]- small forward (Position)
University of Oregon -[EDUCATED_AT]- Maite Cazorla (Player)
University of Oregon -[EDUCATED_AT]- Satou Sabally (Player)
University of Oregon -[EDUCATED_AT -> PLAYS_FOR]- Dallas Wings (Team)
University of Oregon -[EDUCATED_AT -> PLAYS_FOR]- Phoenix Mercury (Team)
University of Oregon -[ATTENDED]- Sabrina Ionescu (Player)
University of Oregon -[ATTENDED -> PLAY

In [ ]:
# Close Connection
conn.close()
print("Selesai.")


Selesai.
